# Regulatory RAG — Dev Log

## Objetivo e papel no pipeline

`core/regulatory_rag` é a **base de conhecimento regulatório local** do
AthenaGov AI: um índice de busca semântica (RAG — Retrieval-Augmented
Generation), 100% local, sobre um corpus ilustrativo de artigos/temas da LGPD
(Lei 13.709/2018).

Este módulo não toma nenhuma decisão de política nem calcula risco — sua única
responsabilidade é, dado um texto de consulta (em linguagem natural), recuperar
os trechos regulatórios mais semanticamente relevantes, com um score de
similaridade. Ele é consumido por dois módulos "de cima" (planejados para a
Onda 2 do roadmap):

- **RIPD Engine** — ao gerar um Relatório de Impacto à Proteção de Dados
  Pessoais, injeta os chunks recuperados no campo
  `RIPDReport.regulatory_context: list[RegulatoryChunk]`, fundamentando
  tecnicamente cada seção do relatório (ex.: citar o Art. 20º ao descrever uma
  decisão automatizada).
- **Governance Copilot** — pode usar `query()` diretamente para responder
  perguntas de um usuário humano sobre "por que este dado exige RIPD?" ou "que
  artigo da LGPD trata de decisão automatizada?", com uma resposta ancorada em
  uma fonte rastreável (em vez de um LLM "alucinando" a partir de conhecimento
  de treinamento genérico).

Funções públicas exportadas: `core.regulatory_rag.build_index()` e
`core.regulatory_rag.query()`.

## Decisões de design

### Por que paráfrase e não o texto oficial da lei

O corpus (`core/regulatory_rag/corpus/*.txt`) **não** contém a transcrição
literal/verbatim dos artigos da LGPD. Cada arquivo é um **resumo/paráfrase
explicitamente identificado como tal**, com um cabeçalho padrão:

> "Resumo ilustrativo do Art. X da LGPD (Lei 13.709/2018) — paráfrase para
> fins de demonstração técnica, não substitui consulta ao texto oficial
> (planalto.gov.br)."

Essa é uma decisão deliberada de engenharia responsável, não uma limitação de
tempo: reproduzir de memória o texto legal exato, palavra por palavra, é um
risco real de desinformação jurídica silenciosa — um erro de transcrição em um
artigo de lei pode ser tomado como autoritativo por quem consome o RAG. Uma
paráfrase claramente rotulada como tal, por outro lado, é auditável (o leitor
sabe que deve conferir a fonte oficial antes de qualquer decisão jurídica real)
e ainda assim captura o conteúdo semântico necessário para a busca funcionar
como demonstração técnica do pipeline RAG. Isso é testado explicitamente em
`test_corpus_loader.py::test_every_chunk_is_explicitly_labeled_as_paraphrase`.
Cobrimos 12 artigos/temas (Art. 5º, 6º, 7º, 9º, 11º, 12º, 18º, 20º, 37º, 38º,
46º, 48º) — o subconjunto mais relevante para governança de IA, não a lei
completa (ver "Limitações" no Handoff Summary).

### Por que `paraphrase-multilingual-MiniLM-L12-v2`

Modelo de embeddings da família sentence-transformers, escolhido por três
critérios que pesam mais que "maior = melhor" neste contexto:

1. **Multilíngue com bom suporte a português** — treinado com paráfrases em
   50+ idiomas, incluindo PT-BR, sem exigir um modelo dedicado a português que
   teria disponibilidade/qualidade menos previsível.
2. **Leve (~118 MB, 384 dimensões)** — roda em CPU, sem GPU, com latência de
   inferência baixa o suficiente para uso interativo (ex.: o Governance
   Copilot respondendo a uma pergunta de usuário em tempo real).
3. **100% local após o primeiro download** — coerente com o requisito de todo
   o V1 do AthenaGov AI: nenhuma chamada a API paga de embeddings (ex.: OpenAI
   embeddings). O único ponto de rede é o download do peso do modelo na
   primeira execução (cacheado depois em `~/.cache/huggingface`).

### Por que ChromaDB

ChromaDB foi escolhido em vez de montar um índice FAISS manual pelos seguintes
motivos: (a) `PersistentClient` já resolve persistência em disco sem código
extra de serialização; (b) a API `collection.query()` já devolve documentos,
metadados e distâncias juntos, o que mapeia diretamente para os campos de
`RegulatoryChunk` (`source`, `article`, `text`, `score`); (c) é a dependência
já listada em `requirements.txt` do projeto (par com `sentence-transformers`),
evitando introduzir uma biblioteca de vetor não planejada. O metadata
(`source`, `article`, `tema`) é armazenado por chunk, permitindo filtrar por
artigo no futuro (V2) sem reindexar.

### Granularidade do chunk

Cada arquivo do corpus é indexado como **um único chunk** (não há split por
parágrafo). Como cada arquivo já é escopado a um artigo/tema específico e tem
tamanho moderado (poucos parágrafos), isso maximiza a chance de o artigo
correto aparecer no top-k para perguntas sobre aquele tema, sem fragmentar o
contexto necessário para a paráfrase fazer sentido isoladamente.

## Setup

In [ ]:
import sys
from pathlib import Path

# Notebook roda a partir de notebooks/ — garante que a raiz do repo (onde ficam
# os pacotes `core` e `shared`) esteja no sys.path.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.regulatory_rag.corpus_loader import load_corpus
from core.regulatory_rag.index import CORPUS_DIR, ModelUnavailableError, build_index, query

print("Imports OK — REPO_ROOT =", REPO_ROOT)
print("Arquivos no corpus:", len(load_corpus(CORPUS_DIR)))

Imports OK — REPO_ROOT = G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)
Arquivos no corpus: 12


## Construindo o índice

`build_index()` lê todo o corpus, gera embeddings com sentence-transformers e
persiste em ChromaDB (`core/regulatory_rag/data/` por padrão). Se o modelo não
puder ser baixado (sem rede), a célula captura `ModelUnavailableError` e
documenta o bloqueio em vez de derrubar o notebook inteiro.

In [ ]:
MODEL_AVAILABLE = True
try:
    n_indexed = build_index()
    print(f"Índice construído: {n_indexed} chunks indexados em core/regulatory_rag/data/")
except ModelUnavailableError as exc:
    MODEL_AVAILABLE = False
    n_indexed = 0
    print("BLOQUEADO — modelo de embeddings indisponível:")
    print(f"  {exc}")

Índice construído: 12 chunks indexados em core/regulatory_rag/data/


## Queries reais

Quatro perguntas de teste em linguagem natural, cobrindo os temas mais
relevantes para governança de IA sob a LGPD: definição de dado sensível,
revisão de decisão automatizada, obrigação de RIPD e direitos do titular.
Para cada uma, mostramos os `RegulatoryChunk` retornados (fonte, artigo,
score) e verificamos que o artigo esperado aparece entre os resultados.

In [ ]:
TEST_QUESTIONS = [
    ("O que é considerado dado pessoal sensível segundo a lei?", "5º"),
    ("Uma decisão automatizada sobre uma pessoa pode ser revisada por um humano?", "20º"),
    ("Quando uma empresa precisa fazer um relatório de impacto (RIPD)?", "38º"),
    ("Quais são os direitos que o titular dos dados pode exercer?", "18º"),
]

if MODEL_AVAILABLE:
    for question, expected_article in TEST_QUESTIONS:
        result = query(question, k=3)
        print(f"=== Pergunta: {question!r} (esperado: Art. {expected_article}) ===")
        for rank, chunk in enumerate(result.chunks, start=1):
            hit = " <-- artigo esperado" if chunk.article == expected_article else ""
            print(f"  #{rank} score={chunk.score:.4f} artigo={chunk.article} fonte={chunk.source}{hit}")
            print(f"      {chunk.text[:140].strip()}...")
        found = expected_article in [c.article for c in result.chunks]
        print(f"  -> artigo esperado no top-{len(result.chunks)}: {found}")
        print()
else:
    print("Pulado — índice não foi construído (modelo de embeddings indisponível).")

=== Pergunta: 'O que é considerado dado pessoal sensível segundo a lei?' (esperado: Art. 5º) ===
  #1 score=0.6339 artigo=11º fonte=art_11_bases_legais_sensivel.txt
      # Resumo ilustrativo do Art. 11º da LGPD (Lei 13.709/2018) — paráfrase para fins de demonstração técnica, não substitui consulta ao texto of...
  #2 score=0.5427 artigo=46º fonte=art_46_seguranca.txt
      # Resumo ilustrativo do Art. 46º da LGPD (Lei 13.709/2018) — paráfrase para fins de demonstração técnica, não substitui consulta ao texto of...
  #3 score=0.5382 artigo=5º fonte=art_5_definicoes.txt <-- artigo esperado
      # Resumo ilustrativo do Art. 5º da LGPD (Lei 13.709/2018) — paráfrase para fins de demonstração técnica, não substitui consulta ao texto ofi...
  -> artigo esperado no top-3: True

=== Pergunta: 'Uma decisão automatizada sobre uma pessoa pode ser revisada por um humano?' (esperado: Art. 20º) ===
  #1 score=0.6327 artigo=20º fonte=art_20_decisoes_automatizadas.txt <-- artigo esperado
      # Resu

## Suíte de testes

Executa `pytest` sobre `core/regulatory_rag/tests` via `subprocess`, usando o
mesmo interpretador do venv do projeto, e mostra o resultado real. Os testes
de `test_corpus_loader.py` não dependem do modelo de embeddings; os de
`test_index.py` são marcados `skipped` (não `failed`) via `ModelUnavailableError`
caso o download do modelo tenha falhado.

In [ ]:
import subprocess
import sys as _sys

proc = subprocess.run(
    [_sys.executable, "-m", "pytest", "core/regulatory_rag/tests", "-v"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(proc.stdout)
print(proc.stderr)
print("return code:", proc.returncode)

============================= test session starts =============================
platform win32 -- Python 3.10.8, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Yuri_\.venvs\athenagov-ai\Scripts\python.exe
cachedir: .pytest_cache
rootdir: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)
plugins: anyio-4.14.2, cov-7.1.0
collecting ... collected 14 items

core/regulatory_rag/tests/test_corpus_loader.py::test_corpus_has_between_10_and_15_files PASSED [  7%]
core/regulatory_rag/tests/test_corpus_loader.py::test_corpus_covers_all_required_articles PASSED [ 14%]
core/regulatory_rag/tests/test_corpus_loader.py::test_every_chunk_has_source_article_and_nonempty_text PASSED [ 21%]
core/regulatory_rag/tests/test_corpus_loader.py::test_every_chunk_is_explicitly_labeled_as_paraphrase PASSED [ 28%]
core/regulatory_rag/tests/test_corpus_loader.py::test_parse_corpus_file_extracts_metadata PASSED [ 35%]
core/regulatory_

## Handoff Summary

**Status desta execução: `done`.** O modelo de embeddings foi baixado com sucesso
(download único, cacheado em `~/.cache/huggingface`) e o índice foi construído e
consultado com dados reais — ver as saídas das células acima. Os 14 testes da
suíte pytest passam integralmente (0 falhas, 0 skips).

### Capacidades entregues

- Corpus local ilustrativo da LGPD com 12 arquivos (`core/regulatory_rag/corpus/*.txt`),
  cobrindo Art. 5º, 6º, 7º, 9º, 11º, 12º, 18º, 20º, 37º, 38º, 46º e 48º — cada um uma
  paráfrase explicitamente identificada como tal (ver "Decisões de design").
- `core/regulatory_rag/corpus_loader.py` — parsing determinístico do corpus
  (`load_corpus`, `parse_corpus_file`), sem dependência de sentence-transformers/chromadb.
- `core/regulatory_rag/index.py` — indexação com embeddings locais
  (sentence-transformers, `paraphrase-multilingual-MiniLM-L12-v2`) e busca semântica
  com ChromaDB persistente (espaço de cosseno), com `ModelUnavailableError` dedicado
  para falhas de download/rede.
- Suíte pytest com 14 testes, todos passando: 7 de parsing do corpus (sempre
  executáveis, sem dependências pesadas) + 7 de indexação/busca semântica, incluindo
  4 perguntas de teste verificando que o artigo esperado aparece entre os top-3
  resultados reais.

### Achado de engenharia durante o desenvolvimento (vale registrar)

A primeira versão do índice rankeava mal: perguntas como "o que é dado sensível?"
retornavam artigos de segurança/incidente (46º/48º) em vez de definições (5º/11º).
Causa raiz: **todo chunk começa com a mesma frase-cabeçalho de paráfrase** ("# Resumo
ilustrativo do Art. X da LGPD... não substitui..."), quase idêntica entre os 12
arquivos — ao ser incluída no texto embedado, essa frase longa e repetida dominava a
similaridade de cosseno e afogava o conteúdo distintivo de cada artigo. A correção
(`index._embedding_text()`) usa, **apenas para gerar o embedding**, o campo `tema` do
chunk (curto e discriminativo) + o corpo sem a linha de disclaimer — mantendo o texto
completo (com o disclaimer) como o `document`/`RegulatoryChunk.text` armazenado e
retornado por `query()`, para que o aviso de paráfrase nunca deixe de aparecer ao
consumidor do RAG. Após a correção, as 4 perguntas de teste passaram a acertar o
artigo esperado no top-3 (ver saída da célula de queries acima).

### Assinatura pública exata

```python
def build_index(persist_dir: str | None = None) -> int:
    """Constrói/reconstrói o índice ChromaDB local a partir de core/regulatory_rag/corpus/.
    Retorna o número de chunks indexados. Levanta ModelUnavailableError se o modelo
    de embeddings não puder ser carregado."""
    ...

def query(text: str, k: int = 3) -> RAGQueryResult:
    """Busca os k chunks regulatórios mais relevantes para `text` no índice
    persistido (core/regulatory_rag/data/ por padrão). Requer build_index() prévio."""
    ...
```

Importável como `from core.regulatory_rag import build_index, query` (ou
`from core.regulatory_rag.index import ...`).

### Como o RIPD Engine / Governance Copilot devem chamar (pseudo-código)

```python
from core.regulatory_rag import query
from shared.schemas import RAGQueryResult

result: RAGQueryResult = query("decisão automatizada sobre uma pessoa", k=3)
regulatory_context = result.chunks  # list[RegulatoryChunk] -> RIPDReport.regulatory_context
```

### Limitações

- O corpus é **ilustrativo e parcial** (12 artigos/temas selecionados por relevância
  para governança de IA) — não é o texto oficial nem a integralidade da LGPD. Todo
  chunk se identifica explicitamente como paráfrase e remete ao texto oficial
  (planalto.gov.br); nenhum conteúdo deste corpus deve ser citado como fonte legal
  primária.
- Cada arquivo do corpus é um único chunk (sem split por parágrafo) — adequado para
  o tamanho atual do corpus, mas não escala para documentos longos sem uma estratégia
  de chunking mais granular.
- A busca é puramente vetorial (semântica) — não há busca híbrida (lexical + vetorial)
  nem re-ranking; para o corpus atual isso já é suficiente (as 4 perguntas de teste
  acertam o top-3), mas um corpus maior tende a se beneficiar de um re-ranker.
- `query()` depende de um índice já construído no diretório padrão
  (`core/regulatory_rag/data/`); não há verificação automática de que o corpus em
  disco e o índice persistido estão sincronizados (reindexação é manual, via
  `build_index()`).

### O que fica para V2

- **GraphRAG** (evolução explícita deste módulo no ROADMAP V2): estruturar o corpus
  como grafo de conhecimento regulatório, capturando relações entre artigos (ex.:
  Art. 20º remete ao dever de transparência do Art. 9º) em vez de apenas similaridade
  vetorial plana.
- **Regulatory Knowledge Graph** (V2): expandir de "corpus de artigos" para um grafo
  de entidades regulatórias (bases legais, direitos, obrigações, sanções) navegável.
- **Regulatory Auto-Update** (V2): pipeline para manter o corpus sincronizado com
  mudanças na lei/regulamentação da ANPD, hoje um processo manual.
- Corpus completo da LGPD (todos os artigos, não apenas o subconjunto priorizado para
  IA) e busca híbrida (lexical + vetorial) com re-ranking.
